# 📊 Análisis de Predicciones vs Valores Reales - Modelo LSTM

Este notebook analiza las predicciones del modelo LSTM comparándolas con los valores reales del conjunto de validación.

**Objetivos:**
- Cargar predicciones del modelo y valores reales
- Calcular métricas de error
- Visualizar predicciones vs valores reales
- Analizar residuales y errores
- Identificar patrones de error

## 1. Importar Librerías

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pyodbc
from datetime import datetime
import os
from dotenv import load_dotenv

# Agregar src/ al path para importar módulos
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Librerías cargadas")

## 2. Cargar Modelo y Datos

In [ ]:
# Cargar variables de entorno
load_dotenv(project_root / '.env')

# Configuración de base de datos
server = os.getenv('SQL_SERVER', 'localhost')
database = os.getenv('SQL_DATABASE', 'MercadoBursatil')
username = os.getenv('SQL_USER', '')
password = os.getenv('SQL_PASSWORD', '')

# Construir connection string
if username and password:
    conn_str = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={username};"
        f"PWD={password}"
    )
else:
    # Autenticación de Windows
    conn_str = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
    )

print(f"✅ Configuración de base de datos: {server}/{database}")

In [ ]:
# Cargar modelo desde database
from ml.shared.persistence.model_registry import ModelRegistry

model_registry = ModelRegistry(
    conn_str=conn_str,
    models_path=project_root / 'models'
)

# Cargar modelo ID 1
model_id = 1
model, model_config = model_registry.load_model(model_id)

print(f"✅ Modelo {model_id} cargado")
print(f"   Configuración: {model_config.to_dict()}")

In [ ]:
# Cargar datos históricos de WALMEX
from ml.shared.data_loaders.ohlcv_loader import OHLCVLoader

# Obtener ticker_id
with pyodbc.connect(conn_str) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT ticker_id FROM stage.cat_tickers WHERE emisora_serie = 'WALMEX*'")
    ticker_id = cursor.fetchone()[0]

print(f"✅ Ticker ID: {ticker_id}")

# Cargar datos
data_loader = OHLCVLoader(
    ticker_id=ticker_id,
    ticker_symbol='WALMEX*',
    server=server,
    database=database,
    username=username,
    password=password
)

# Cargar últimos 3 años
from datetime import timedelta
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=1095)).strftime('%Y-%m-%d')

data = data_loader.load_data(start_date=start_date, end_date=end_date)
data = data.reset_index()

print(f"✅ Datos cargados: {len(data)} registros")
print(f"   Rango: {data['fecha'].min()} a {data['fecha'].max()}")

## 3. Preparar Datos y Generar Predicciones

Aplicamos el mismo preprocesamiento que durante el entrenamiento:
1. Feature engineering (26 indicadores técnicos)
2. Escalado (MinMaxScaler)
3. Generación de secuencias (60 días de lookback)
4. Split temporal (train/val/test)

In [ ]:
# 1. Feature Engineering
from ml.shared.preprocessing.feature_engineer import FeatureEngineer

feature_engineer = FeatureEngineer(data)
data_with_features = feature_engineer.add_all_features()

print(f"✅ Features creados: {len(data_with_features.columns)} columnas")
print(f"   Datos después de features: {len(data_with_features)} registros")
print(f"   Columnas: {list(data_with_features.columns)}")

In [ ]:
# 2. Escalado
from ml.shared.preprocessing.data_scaler import DataScaler

# Reorganizar para que 'close' esté primero (es el target)
feature_cols = ['close'] + [col for col in data_with_features.columns if col not in ['close', 'fecha']]
data_ordered = data_with_features[feature_cols].copy()

scaler = DataScaler(scaler_type='minmax')
data_scaled = scaler.fit_transform(data_ordered)

print(f"✅ Datos escalados: {data_scaled.shape}")
print(f"   Min: {data_scaled.min():.4f}, Max: {data_scaled.max():.4f}")

In [ ]:
# 3. Generación de Secuencias
from ml.shared.preprocessing.sequence_generator import SequenceGenerator

sequence_gen = SequenceGenerator(
    sequence_length=60,
    prediction_horizon=5
)

X, y = sequence_gen.create_sequences(data=data_scaled, target_column_idx=0)

print(f"✅ Secuencias generadas:")
print(f"   X shape: {X.shape} (samples, sequence_length, features)")
print(f"   y shape: {y.shape} (samples, prediction_horizon)")

In [ ]:
# 4. Split Temporal (Train/Val/Test)
from ml.shared.preprocessing.temporal_splitter import TemporalSplitter

splitter = TemporalSplitter(
    test_size=30,
    val_size=30
)

X_train, X_val, X_test, y_train, y_val, y_test = splitter.split_arrays(X, y)

print(f"✅ Datos divididos:")
print(f"   Train: {X_train.shape[0]} secuencias")
print(f"   Val:   {X_val.shape[0]} secuencias")
print(f"   Test:  {X_test.shape[0]} secuencias")

In [ ]:
# 5. Generar Predicciones
y_val_pred_scaled = model.predict(X_val)
y_test_pred_scaled = model.predict(X_test)

print(f"✅ Predicciones generadas:")
print(f"   Val predictions: {y_val_pred_scaled.shape}")
print(f"   Test predictions: {y_test_pred_scaled.shape}")

In [ ]:
# 6. Desnormalizar predicciones y valores reales
# Necesitamos invertir el escalado solo para la columna 'close' (índice 0)

# Obtener el scaler de la primera columna (close)
close_scaler = scaler.scaler

# Desnormalizar valores reales
y_val_real = close_scaler.inverse_transform(y_val.reshape(-1, 1)).reshape(y_val.shape)
y_test_real = close_scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)

# Desnormalizar predicciones
y_val_pred = close_scaler.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).reshape(y_val_pred_scaled.shape)
y_test_pred = close_scaler.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).reshape(y_test_pred_scaled.shape)

print(f"✅ Valores desnormalizados:")
print(f"   Val real rango: ${y_val_real.min():.2f} - ${y_val_real.max():.2f}")
print(f"   Val pred rango: ${y_val_pred.min():.2f} - ${y_val_pred.max():.2f}")
print(f"   Test real rango: ${y_test_real.min():.2f} - ${y_test_real.max():.2f}")
print(f"   Test pred rango: ${y_test_pred.min():.2f} - ${y_test_pred.max():.2f}")

## 4. Calcular Métricas de Error

In [ ]:
# Calcular métricas para cada horizonte de predicción (1-5 días)
def calculate_metrics_per_horizon(y_true, y_pred, set_name):
    """Calcular métricas para cada día del horizonte"""
    results = []
    
    for day in range(y_true.shape[1]):
        mae = mean_absolute_error(y_true[:, day], y_pred[:, day])
        rmse = np.sqrt(mean_squared_error(y_true[:, day], y_pred[:, day]))
        r2 = r2_score(y_true[:, day], y_pred[:, day])
        mape = np.mean(np.abs((y_true[:, day] - y_pred[:, day]) / y_true[:, day])) * 100
        
        results.append({
            'Set': set_name,
            'Día': day + 1,
            'MAE': mae,
            'RMSE': rmse,
            'R²': r2,
            'MAPE (%)': mape
        })
    
    return pd.DataFrame(results)

# Calcular métricas
metrics_val = calculate_metrics_per_horizon(y_val_real, y_val_pred, 'Validación')
metrics_test = calculate_metrics_per_horizon(y_test_real, y_test_pred, 'Test')

# Combinar resultados
all_metrics = pd.concat([metrics_val, metrics_test], ignore_index=True)

print("📊 MÉTRICAS POR HORIZONTE DE PREDICCIÓN")
print("=" * 80)
print(all_metrics.to_string(index=False))
print("=" * 80)

## 5. Visualizaciones: Predicciones vs Valores Reales

In [ ]:
# 1. Scatter Plot: Predicciones vs Valores Reales (Día 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Validación
axes[0].scatter(y_val_real[:, 0], y_val_pred[:, 0], alpha=0.6, s=50)
axes[0].plot([y_val_real[:, 0].min(), y_val_real[:, 0].max()], 
             [y_val_real[:, 0].min(), y_val_real[:, 0].max()], 
             'r--', lw=2, label='Línea perfecta')
axes[0].set_xlabel('Valores Reales ($)', fontsize=12)
axes[0].set_ylabel('Predicciones ($)', fontsize=12)
axes[0].set_title('Validación - Día 1 (predicción próximo día)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test
axes[1].scatter(y_test_real[:, 0], y_test_pred[:, 0], alpha=0.6, s=50, color='orange')
axes[1].plot([y_test_real[:, 0].min(), y_test_real[:, 0].max()], 
             [y_test_real[:, 0].min(), y_test_real[:, 0].max()], 
             'r--', lw=2, label='Línea perfecta')
axes[1].set_xlabel('Valores Reales ($)', fontsize=12)
axes[1].set_ylabel('Predicciones ($)', fontsize=12)
axes[1].set_title('Test - Día 1 (predicción próximo día)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 2. Series de Tiempo: Predicciones vs Reales (primeras 50 muestras de validación)
n_samples = min(50, len(y_val_real))

fig, axes = plt.subplots(5, 1, figsize=(15, 15))

for day in range(5):
    axes[day].plot(range(n_samples), y_val_real[:n_samples, day], 
                   'o-', label='Real', linewidth=2, markersize=6)
    axes[day].plot(range(n_samples), y_val_pred[:n_samples, day], 
                   's-', label='Predicción', linewidth=2, markersize=6, alpha=0.7)
    axes[day].set_xlabel('Muestra', fontsize=11)
    axes[day].set_ylabel('Precio ($)', fontsize=11)
    axes[day].set_title(f'Día {day+1} - Comparación Real vs Predicción (Validación)', 
                        fontsize=12, fontweight='bold')
    axes[day].legend(loc='best')
    axes[day].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Análisis de Errores y Residuales

In [ ]:
# Calcular residuales (errores)
residuals_val = y_val_pred - y_val_real
residuals_test = y_test_pred - y_test_real

# 1. Distribución de Residuales
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for day in range(5):
    row = day // 3
    col = day % 3
    
    # Validación
    axes[row, col].hist(residuals_val[:, day], bins=30, alpha=0.6, color='blue', 
                        edgecolor='black', label='Validación')
    # Test
    axes[row, col].hist(residuals_test[:, day], bins=30, alpha=0.6, color='orange', 
                        edgecolor='black', label='Test')
    
    axes[row, col].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Error = 0')
    axes[row, col].set_xlabel('Error ($)', fontsize=11)
    axes[row, col].set_ylabel('Frecuencia', fontsize=11)
    axes[row, col].set_title(f'Distribución de Errores - Día {day+1}', fontsize=12, fontweight='bold')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

# Remover subplot vacío
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

In [ ]:
# 2. Errores a lo largo del tiempo (Validación)
fig, axes = plt.subplots(5, 1, figsize=(15, 15))

for day in range(5):
    axes[day].plot(residuals_val[:, day], 'o-', alpha=0.6, markersize=4)
    axes[day].axhline(y=0, color='red', linestyle='--', linewidth=2)
    axes[day].fill_between(range(len(residuals_val)), 
                           residuals_val[:, day].mean() - residuals_val[:, day].std(),
                           residuals_val[:, day].mean() + residuals_val[:, day].std(),
                           alpha=0.2, color='gray', label='±1 std')
    axes[day].set_xlabel('Muestra', fontsize=11)
    axes[day].set_ylabel('Error ($)', fontsize=11)
    axes[day].set_title(f'Errores a lo Largo del Tiempo - Día {day+1} (Validación)', 
                        fontsize=12, fontweight='bold')
    axes[day].legend()
    axes[day].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Estadísticas de Error por Horizonte

In [ ]:
# Estadísticas de residuales por horizonte
error_stats_val = pd.DataFrame({
    'Día': range(1, 6),
    'Mean Error ($)': [residuals_val[:, i].mean() for i in range(5)],
    'Std Error ($)': [residuals_val[:, i].std() for i in range(5)],
    'Min Error ($)': [residuals_val[:, i].min() for i in range(5)],
    'Max Error ($)': [residuals_val[:, i].max() for i in range(5)],
    'Median Error ($)': [np.median(residuals_val[:, i]) for i in range(5)]
})

error_stats_test = pd.DataFrame({
    'Día': range(1, 6),
    'Mean Error ($)': [residuals_test[:, i].mean() for i in range(5)],
    'Std Error ($)': [residuals_test[:, i].std() for i in range(5)],
    'Min Error ($)': [residuals_test[:, i].min() for i in range(5)],
    'Max Error ($)': [residuals_test[:, i].max() for i in range(5)],
    'Median Error ($)': [np.median(residuals_test[:, i]) for i in range(5)]
})

print("📊 ESTADÍSTICAS DE ERROR - VALIDACIÓN")
print("=" * 100)
print(error_stats_val.to_string(index=False))
print("\n📊 ESTADÍSTICAS DE ERROR - TEST")
print("=" * 100)
print(error_stats_test.to_string(index=False))

In [ ]:
# Visualizar cómo crecen los errores con el horizonte
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RMSE por horizonte
rmse_by_horizon_val = [np.sqrt(mean_squared_error(y_val_real[:, i], y_val_pred[:, i])) for i in range(5)]
rmse_by_horizon_test = [np.sqrt(mean_squared_error(y_test_real[:, i], y_test_pred[:, i])) for i in range(5)]

axes[0].plot(range(1, 6), rmse_by_horizon_val, 'o-', linewidth=2, markersize=8, label='Validación')
axes[0].plot(range(1, 6), rmse_by_horizon_test, 's-', linewidth=2, markersize=8, label='Test')
axes[0].set_xlabel('Horizonte (días)', fontsize=12)
axes[0].set_ylabel('RMSE ($)', fontsize=12)
axes[0].set_title('RMSE por Horizonte de Predicción', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAPE por horizonte
mape_by_horizon_val = [np.mean(np.abs((y_val_real[:, i] - y_val_pred[:, i]) / y_val_real[:, i])) * 100 for i in range(5)]
mape_by_horizon_test = [np.mean(np.abs((y_test_real[:, i] - y_test_pred[:, i]) / y_test_real[:, i])) * 100 for i in range(5)]

axes[1].plot(range(1, 6), mape_by_horizon_val, 'o-', linewidth=2, markersize=8, label='Validación')
axes[1].plot(range(1, 6), mape_by_horizon_test, 's-', linewidth=2, markersize=8, label='Test')
axes[1].set_xlabel('Horizonte (días)', fontsize=12)
axes[1].set_ylabel('MAPE (%)', fontsize=12)
axes[1].set_title('MAPE por Horizonte de Predicción', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Conclusiones

**Interpretación de Resultados:**

1. **Precisión del Modelo:** Observa los valores de RMSE, MAE y R². Valores bajos de error y R² cercano a 1 indican buen ajuste.

2. **Horizonte de Predicción:** Verifica si los errores aumentan conforme el horizonte se aleja (Día 1 vs Día 5). Es normal que predicciones más lejanas tengan mayor error.

3. **Distribución de Errores:** Los residuales deben estar centrados en 0 y seguir distribución normal. Si hay sesgo (mean ≠ 0), el modelo subestima o sobreestima sistemáticamente.

4. **Validación vs Test:** Si las métricas son muy diferentes entre validación y test, puede haber overfitting.

5. **Data Leakage:** Si los resultados son **perfectos** (R² = 1.0, RMSE = 0), revisa que no haya data leakage en el proceso de entrenamiento.